### Import libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

import matplotlib as mpl
mpl.rcParams["font.family"] = "Nimbus Roman"
from matplotlib.collections import LineCollection

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))

from ml_force.models import MorrisLecarBlockGPU, z_transform
from ml_force.supervisors import LorenzAttractor

In [2]:
T = 500
dt = 1e-2
t = np.arange(0, T, dt)

### Run Models

#### Preparing functions

In [ ]:
x = LorenzAttractor(T, dt, tau=0.04).generate(transient_time=200)
# x = np.sin(2 * 1 * np.pi * t / 1000)
# x = x.reshape(-1, 1)
x = x.T
x = z_transform(x)

print(x.shape)

In [ ]:
plt.plot(t, x[:, 0])

#### Run Block model

In [ ]:
## TORCH SETUP
seed = 2024
np.random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(f"Using Device <{device}> for PyTorch computations...\n")
torch.random.manual_seed(seed)
torch.cuda.random.manual_seed(seed)

In [6]:
N = 300
# input current for I and E neurons
Ie = 85
Ii = 85
current = np.ones((N, 1))
middle = N // 2
current[:middle] *= Ie  # NE bias
current[middle:] *= Ii  # NI bias

In [7]:
rls_start = 2100
rls_stop = 1600
rls_step = 20

render_params = {"rls_start": rls_start,
                    "rls_stop": rls_stop,
                    "rls_step": rls_step,
                    "live_plot": False,
                    "n_neurons": 10,
                    "plt_interval": 300,
                    "save_all": False}


In [8]:
x = np.zeros(shape=x.shape, dtype=float)
block = MorrisLecarBlockGPU(supervisor=x, T=T, dt=dt, N=N, BIAS=current, 
                            gbar=5, Q=200, l=1e-5, device=device)

In [ ]:

_, v_trace, decoder = block.render(**render_params)

In [10]:
block_mod = MorrisLecarBlockGPU(supervisor=x, T=T, dt=dt, N=N, BIAS=current, 
                            # a_r=2.2, a_d=.04,
                            # v2=9, v4=15,
                            # g_L=2, g_K=8, g_Ca=10,
                            C=5,
                            gbar=5, Q=200, l=1e-5, device=device)

In [ ]:
_, v_trace_mod, decoder_mod = block_mod.render(**render_params)

In [ ]:


voltage_trace_np = v_trace.numpy()
step = 10

fig, ax = plt.subplots(figsize=(15, 8), nrows=2)

# The original model
for i in range(voltage_trace_np.shape[1]):
    signal = voltage_trace_np[::step, i]
    minim = np.min(signal)
    maxim = np.max(signal)
    signal = (signal - minim) / (maxim - minim) + i+1
    ax[0].plot(t[::step], signal, lw=1)
ax[0].grid(alpha=.5)
ax[0].set_xlabel("Time [ms]")
ax[0].set_ylabel("Voltage Trace (normalized)")

vt_mod = v_trace_mod.numpy()
# The modified model
for i in range(vt_mod.shape[1]):
    signal = vt_mod[::step, i]
    minim = np.min(signal)
    maxim = np.max(signal)
    signal = (signal - minim) / (maxim - minim) + i+1
    ax[1].plot(t[::step], signal, lw=1)
ax[1].grid(alpha=.5)
ax[1].set_xlabel("Time [ms]")
ax[1].set_ylabel("Voltage Trace (normalized)")

plt.suptitle(f"N = {block._N}, Ie = {Ie}, Ii = {Ii}", fontsize=20)
# plt.savefig("img/initial_chaos_voltage_trace2.jpg", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ax 1
shift = 105
midpoint = block._N // 2
e_ind = shift
i_ind = midpoint + shift
# e_ind = 267
# i_ind = 252
start = 10000
ve_store = voltage_trace_np[start::step, e_ind]
vi_store = voltage_trace_np[start::step, i_ind]
points = np.array([ve_store, vi_store]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
norm = mpl.colors.Normalize(vmin=t.min(), vmax=t.max())
lc = LineCollection(segments, norm=norm, cmap='bwr', linewidth=1)
lc.set_array(t[start::step])
ax[1].add_collection(lc)
ax[1].autoscale()
# ax[1].plot(ve_store, vi_store, lw=.5, norm=norm, cmap='inferno')
ax[1].grid(alpha=.5)
ax[1].set_xlabel(f"neuron {e_ind}")
ax[1].set_ylabel(f"neuron {i_ind}")
cbar = plt.colorbar(lc)
cbar.set_label('Time [ms]')

In [ ]:
rands

In [ ]:
from scipy.signal import find_peaks

from ml_force import LorenzAttractor


# Return Map

signal = LorenzAttractor(T, dt, 0.01).generate(2000)[2]
print(signal.shape)
signal = z_transform(signal)
peaks = find_peaks(signal)[0]

plt.plot(signal[peaks[:-1]], signal[peaks[1:]], 'go')
plt.show()

In [9]:
def return_map(signal:np.ndarray):
    peaks = find_peaks(signal)[0]
    plt.plot(signal[peaks[:-1]], signal[peaks[1:]], 'go')
    plt.show()

In [ ]:
return_map(ve_store)

In [ ]:
clr = plt.pcolormesh(block.w, cmap="viridis")
plt.colorbar(clr)
plt.title(f"Adjacency Matrix of the Coupled Reservoir, N={block._N}")
plt.savefig("img/weight_matrix.jpg", dpi=300, bbox_inches='tight')
plt.show()